In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import json
from pathlib import Path
plt.style.use('seaborn-v0_8-whitegrid')
PROCESSED = Path('data/processed')
print('Imports OK')

In [ ]:
full = pd.read_parquet(PROCESSED / 'full.parquet')
train = pd.read_parquet(PROCESSED / 'train.parquet')
val   = pd.read_parquet(PROCESSED / 'val.parquet')
test  = pd.read_parquet(PROCESSED / 'test.parquet')
item_meta = pd.read_parquet(PROCESSED / 'item_metadata.parquet')
user_meta = pd.read_parquet(PROCESSED / 'user_metadata.parquet')
stats = json.load(open(PROCESSED / 'dataset_stats.json'))
print(f'Users: {full.user_id.nunique()} | Items: {full.item_id.nunique()} | Ratings: {len(full):,}')

In [ ]:
summary = pd.DataFrame({
    'Split': ['Train', 'Val', 'Test', 'Total'],
    'Interactions': [len(train), len(val), len(test), len(full)],
    'Users': [train.user_id.nunique(), val.user_id.nunique(),
              test.user_id.nunique(), full.user_id.nunique()],
    'Items': [train.item_id.nunique(), val.item_id.nunique(),
              test.item_id.nunique(), full.item_id.nunique()],
})
print(summary.to_string(index=False))
print(f'Avg ratings/user: {len(full)/full.user_id.nunique():.1f}')

In [ ]:
user_counts = train.groupby('user_id').size()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(user_counts, bins=50, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Ratings per user'); axes[0].set_ylabel('Count')
axes[0].set_title('Ratings per User Distribution')
axes[1].hist(user_counts.clip(upper=50), bins=50, color='coral', edgecolor='white')
axes[1].set_xlabel('Ratings per user (clipped at 50)'); axes[1].set_ylabel('Count')
axes[1].set_title('Ratings per User (Zoomed)')
plt.tight_layout()
Path('artifacts').mkdir(exist_ok=True)
plt.savefig('artifacts/eda_user_dist.png', dpi=120)
plt.show()

In [ ]:
item_counts = train.groupby('item_id').size().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(range(len(item_counts)), item_counts.values, color='steelblue', linewidth=0.8)
ax.set_xlabel('Item rank (by popularity)'); ax.set_ylabel('Interaction count')
ax.set_title('Item Popularity \u2014 Long Tail Distribution')
ax.set_yscale('log')
top20_pct = item_counts.head(int(len(item_counts)*0.2)).sum() / item_counts.sum()
ax.axvline(int(len(item_counts)*0.2), color='red', linestyle='--',
           label=f'Top 20% items = {top20_pct:.1%} of interactions')
ax.legend()
plt.tight_layout()
plt.savefig('artifacts/eda_item_longtail.png', dpi=120)
plt.show()
print(f'Top 20% items cover {top20_pct:.1%} of all interactions (long-tail confirmed)')

In [ ]:
age_labels = {0:'<18', 1:'18-24', 2:'25-34', 3:'35-44', 4:'45-49', 5:'50-55', 6:'56+'}
age_counts = user_meta['age_enc'].map(age_labels).value_counts()
gender_counts = user_meta['gender_enc'].map({0:'Female', 1:'Male'}).value_counts()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(gender_counts.index, gender_counts.values, color=['coral', 'steelblue'])
axes[0].set_title('Gender Distribution'); axes[0].set_ylabel('Users')
age_order = ['<18','18-24','25-34','35-44','45-49','50-55','56+']
axes[1].bar(age_order, [age_counts.get(a,0) for a in age_order], color='steelblue')
axes[1].set_title('Age Distribution'); axes[1].set_ylabel('Users')
plt.tight_layout()
plt.savefig('artifacts/eda_demographics.png', dpi=120)
plt.show()

In [ ]:
genre_cols = [c for c in item_meta.columns if c.startswith('genre_') and c != 'genre_vector']
if genre_cols:
    genre_counts = item_meta[genre_cols].sum().sort_values(ascending=False)
    genre_counts.index = [c.replace('genre_','').replace('_',' ').title() for c in genre_counts.index]
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.barh(genre_counts.index[:12], genre_counts.values[:12], color='steelblue')
    ax.set_title('Top 12 Genres in ML-1M'); ax.set_xlabel('Number of movies')
    plt.tight_layout()
    plt.savefig('artifacts/eda_genres.png', dpi=120)
    plt.show()
else:
    print('No per-genre columns found in item_metadata')

In [ ]:
n_users = full.user_id.nunique()
n_items = full.item_id.nunique()
sparsity = 1 - len(full) / (n_users * n_items)
print(f'Matrix sparsity: {sparsity:.4%}')
print(f'Avg ratings/user: {len(full)/n_users:.1f}')
print(f'Avg ratings/item: {len(full)/n_items:.1f}')
print(f'Leave-one-out val/test: {len(val)} users each')
print('EDA complete. Plots saved to artifacts/')